In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
%sql
select * from fintech360.silver.emi_data limit 5;

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================================================
# 1. READ SILVER TABLE
# =========================================================

source_df = spark.read.table("fintech360.silver.emi_data")

# Normalize values used for validation.
# Keep original columns unchanged; use helper columns for checks.

df = (source_df
    .withColumn("_emi_id", F.col("emi_id").cast("long"))
    .withColumn("_loan_id", F.col("loan_id").cast("long"))
    .withColumn("_installment_number", F.col("installment_number").cast("int"))
    .withColumn("_emi_amount", F.col("emi_amount").cast("double"))
    .withColumn("_paid_amount", F.col("paid_amount").cast("double"))
    .withColumn("_days_past_due", F.col("days_past_due").cast("int"))
    .withColumn("_late_fee_amount", F.col("late_fee_amount").cast("double"))
    .withColumn("_status", F.upper(F.trim(F.col("emi_status").cast("string"))))
    .withColumn("_mandate", F.upper(F.trim(F.col("mandate_type").cast("string"))))
    .withColumn("_payment_method", F.upper(F.trim(F.col("payment_method").cast("string"))))
)

# =========================================================
# 2. CONVERT UNIX EPOCH SECONDS TO TIMESTAMPS FOR VALIDATION
# =========================================================

timestamp_columns = ["due_date", "paid_at", "created_at", "updated_at"]

for c in timestamp_columns:
    df = df.withColumn(
        f"_dq_{c}",
        F.when(
            F.col(c).isNotNull() &
            (F.lower(F.trim(F.col(c).cast("string"))) != "null"),
            F.to_timestamp(
                F.from_unixtime(F.col(c).cast("long"))
            )
        )
    )

# =========================================================
# 3. IDENTIFY DUPLICATE EMI IDS
# Quarantine every row having a duplicated non-null EMI ID.
# =========================================================

duplicate_window = Window.partitionBy("_emi_id")

df = df.withColumn(
    "_duplicate_count",
    F.when(
        F.col("_emi_id").isNotNull(),
        F.count(F.lit(1)).over(duplicate_window)
    ).otherwise(F.lit(0))
)

# =========================================================
# 4. DEFINE DATA QUALITY RULES
# Each failed rule adds a reason to dq_failed_reasons.
# =========================================================

allowed_statuses = ["PENDING", "PAID", "OVERDUE"]

# Extend these lists if your business supports other values.
allowed_mandates = ["CARD_AUTOPAY", "UPI_AUTOPAY", "NACH"]

allowed_payment_methods = ["CASH", "BANK_TRANSFER", "CARD", "UPI"]

rules = [
    (
        F.col("emi_id").isNull() | F.col("_emi_id").isNull(),
        "EMI_ID_NULL_OR_INVALID"
    ),
    (
        F.col("loan_id").isNull() | F.col("_loan_id").isNull(),
        "LOAN_ID_NULL_OR_INVALID"
    ),
    (
        F.col("_duplicate_count") > 1,
        "DUPLICATE_EMI_ID"
    ),
    (
        F.col("installment_number").isNull() |
        F.col("_installment_number").isNull() |
        (F.col("_installment_number") <= 0),
        "INVALID_INSTALLMENT_NUMBER"
    ),
    (
        F.col("emi_amount").isNull() |
        F.col("_emi_amount").isNull() |
        F.isnan(F.col("_emi_amount")) |
        (F.col("_emi_amount") <= 0),
        "INVALID_EMI_AMOUNT"
    ),
    (
        F.col("paid_amount").isNull() |
        F.col("_paid_amount").isNull() |
        F.isnan(F.col("_paid_amount")) |
        (F.col("_paid_amount") < 0) |
        (F.col("_paid_amount") > F.col("_emi_amount")),
        "INVALID_PAID_AMOUNT"
    ),
    (
        F.col("days_past_due").isNull() |
        F.col("_days_past_due").isNull() |
        (F.col("_days_past_due") < 0),
        "INVALID_DAYS_PAST_DUE"
    ),
    (
        F.col("late_fee_amount").isNull() |
        F.col("_late_fee_amount").isNull() |
        F.isnan(F.col("_late_fee_amount")) |
        (F.col("_late_fee_amount") < 0),
        "INVALID_LATE_FEE"
    ),
    (
        F.col("_status").isNull() |
        (~F.col("_status").isin(allowed_statuses)),
        "INVALID_EMI_STATUS"
    ),
    (
        (F.col("_status") == "PAID") &
        (
            F.col("_dq_paid_at").isNull() |
            (
                F.abs(
                    F.col("_paid_amount") - F.col("_emi_amount")
                ) > F.lit(0.01)
            )
        ),
        "PAID_EMI_INCONSISTENT"
    ),
    (
        (F.col("_status") == "OVERDUE") &
        (F.col("_days_past_due") <= 0),
        "OVERDUE_WITHOUT_POSITIVE_DPD"
    ),
    (
        F.col("_dq_due_date").isNull(),
        "INVALID_DUE_DATE"
    ),
    (
        F.col("_dq_created_at").isNull(),
        "INVALID_CREATED_AT"
    ),
    (
        F.col("_dq_updated_at").isNull(),
        "INVALID_UPDATED_AT"
    ),
    (
        F.col("_dq_created_at").isNotNull() &
        F.col("_dq_updated_at").isNotNull() &
        (F.col("_dq_updated_at") < F.col("_dq_created_at")),
        "UPDATED_BEFORE_CREATED"
    ),
    (
        (F.col("_status") == "PAID") &
        F.col("_dq_paid_at").isNull(),
        "PAID_WITHOUT_PAYMENT_DATE"
    ),
    (
        F.col("_mandate").isNull() |
        (~F.col("_mandate").isin(allowed_mandates)),
        "INVALID_MANDATE_TYPE"
    ),
    (
        F.col("_payment_method").isNull() |
        (~F.col("_payment_method").isin(allowed_payment_methods)),
        "INVALID_PAYMENT_METHOD"
    )
]

# =========================================================
# 5. BUILD THE QUARANTINE REASONS
# concat_ws ignores null values, so only failed rules appear.
# =========================================================

df = df.withColumn(
    "dq_failed_reasons",
    F.concat_ws(
        " | ",
        *[
            F.when(condition, F.lit(reason))
            for condition, reason in rules
        ]
    )
)

# =========================================================
# 6. CREATE PASSED AND QUARANTINED DATAFRAMES
# =========================================================

helper_columns = [
    "_emi_id", "_loan_id", "_installment_number",
    "_emi_amount", "_paid_amount", "_days_past_due",
    "_late_fee_amount", "_status", "_mandate",
    "_payment_method", "_duplicate_count",
    *[f"_dq_{c}" for c in timestamp_columns]
]

original_columns = source_df.columns

emi_passed = (
    df.filter(F.col("dq_failed_reasons") == "")
      .select(*original_columns)
)

quarantine_data = (
    df.filter(F.col("dq_failed_reasons") != "")
      .drop(*helper_columns)
)

# =========================================================
# 7. VALIDATE THE GATE RESULTS
# =========================================================

source_count = source_df.count()
passed_count = emi_passed.count()
quarantine_count = quarantine_data.count()

print(f"Source records     : {source_count}")
print(f"Passed records     : {passed_count}")
print(f"Quarantined records: {quarantine_count}")
print(f"Reconciliation     : {passed_count + quarantine_count}")

# Inspect quarantined records and their failure reasons.
display(
    quarantine_data.select(
        "emi_id",
        "loan_id",
        "emi_status",
        "emi_amount",
        "paid_amount",
        "days_past_due",
        "dq_failed_reasons"
    )
)

# Only this DataFrame should move to the next layer.
display(emi_passed)


In [0]:
# Passed records
(emi_passed.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.data_quality_check.emi_data"))

# Quarantined records, including dq_failed_reasons
(quarantine_data.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.quarantine_data.emi_data"))